In [14]:
import re
import pandas as pd
import numpy as np

In [15]:
import spacy

nlp = spacy.load("pt_core_news_md")

In [16]:
def converter_expressao_para_regex(expressao_booleana):
    """Conversor automático de expressões (A OU B) E (C OU D) para Regex Lookahead"""
    if not isinstance(expressao_booleana, str) or not expressao_booleana.strip():
        return ""

    # 1. Encontra tudo o que está dentro dos parênteses (...)
    # Isso isola os grupos que estão separados por " E "
    grupos_parenteses = re.findall(r"\((.*?)\)", expressao_booleana)

    # Se a estrutura não tiver parênteses, tenta quebrar direto pelo " E "
    if not grupos_parenteses:
        grupos_parenteses = expressao_booleana.split(" E ")

    lookaheads = []

    for grupo in grupos_parenteses:
        # Quebra os termos internos pelo " OU "
        termos = [termo.strip() for termo in grupo.split(" OU ") if termo.strip()]

        if not termos:
            continue

        # Ordena do maior termo para o menor (evita que 'DP' engula 'Defensoria Pública')
        termos_ordenados = sorted(termos, key=len, reverse=True)

        # Escapa caracteres especiais que possam existir no texto (como pontos ou interrogações)
        termos_escapados = "|".join(
            [re.escape(termo) for termo in termos_ordenados]
        )

        # Monta o Lookahead para este grupo específico garantindo as fronteiras de palavra (\b)
        # O .*? garante que ele ache o termo em qualquer parte do texto longo
        lookahead_grupo = f"(?=.*?(?:{termos_escapados}))"
        lookaheads.append(lookahead_grupo)

    # Une todos os lookaheads. Para a Regex dar MATCH, TODOS os grupos precisam ser verdadeiros (Lógica "E")
    regex_final = "".join(lookaheads)
    return regex_final


In [17]:
def aplicar_regex_no_dataframe(df, df_regex):
    """Aplica as expressões regulares de df_regex no df e lista os temas encontrados."""
    # Inicializa a coluna com listas vazias para cada linha
    df = df.copy()
    df.loc[:, "temas_encontrados"] = [[] for _ in range(len(df))]

    for _, row in df_regex.iterrows():
        regex = row["REGEX"]
        tema_codigo = row["TEMA CÓDIGO"]  # Nome exato da coluna do seu CSV

        # Pula se a regex estiver vazia
        if regex=="NAN":
            continue

        # Cria uma máscara booleana para as linhas que dão match
        matches = df["inteiro_teor_lematizado"].str.contains(
            regex, flags=re.IGNORECASE, regex=True, na=False
        )
        total_matches = df["inteiro_teor_lematizado"].str.contains(
            regex, flags=re.IGNORECASE, regex=True, na=False
        ).sum()
        print("Total de palavras:", total_matches)
        # Para as linhas onde deu Match, adiciona o código do tema na lista
        df.loc[matches, "temas_encontrados"] = df.loc[
            matches, "temas_encontrados"
        ].apply(lambda x: x + [tema_codigo])

    return df

In [18]:
def aplicar_regex_flexivel_no_dataframe(df, df_regex, corte_porcentagem=30.0):
    """
    Aplica as expressões regulares de forma flexível, tratando caracteres especiais
    e corrigindo erros de compilação de Regex.
    """
    df = df.copy()
    
    # Inicializa as colunas de resultados
    df["temas_encontrados"] = [[] for _ in range(len(df))]
    df["termos_capturados"] = [[] for _ in range(len(df))]
    df["acertou_tema"] = False
    
    for _, row in df_regex.iterrows():
        regex_completa = row["PALAVRAS_LEMATIZADAS_REGEX"]
        tema_codigo = row["TEMA CÓDIGO"]


        if regex_completa == "NAN" or pd.isna(regex_completa):
            continue

        # 1. Extração robusta dos blocos de termos internos
        # Captura o conteúdo de dentro dos parênteses dos Lookaheads
        blocos_sujos = re.findall(r"\?=\.\*\?\(([^)]+)\)", regex_completa)
        if not blocos_sujos:
            blocos_sujos = re.findall(r"\(([^)]+)\)", regex_completa)
            
        # Limpa e valida os blocos extraídos
        blocos = []
        for b in blocos_sujos:
            # Remove escapes manuais de parênteses internos que possam quebrar o motor do re.findall
            b_limpo = b.replace(r'\(', '(').replace(r'\)', ')')
            blocos.append(b_limpo)
            
        total_blocos = len(blocos)
        if total_blocos == 0:
            continue

        #print(f"Processando Tema {tema_codigo} (Possui {total_blocos} grupos)...")

        # 2. Avalia linha por linha com tratamento de erro integrado
        for idx, texto in df["inteiro_teor_lematizado"].items():
            if pd.isna(texto) or not isinstance(texto, str):
                continue
                
            blocos_com_match = 0
            termos_da_linha = []

            for bloco in blocos:
                try:
                    # Tenta rodar o bloco original
                    matches_bloco = re.findall(bloco, texto, flags=re.IGNORECASE)
                except re.error:
                    # SE o bloco tiver caracteres especiais quebrados (ex: parênteses soltos),
                    # o Python escapará o bloco dinamicamente para tratá-lo como texto puro literal
                    bloco_seguro = "|".join([re.escape(termo.strip().replace('\\', '')) for termo in bloco.split('|')])
                    matches_bloco = re.findall(bloco_seguro, texto, flags=re.IGNORECASE)
                
                if matches_bloco:
                    blocos_com_match += 1
                    # Garante que salve strings (caso retorne tuplas de grupos)
                    termos_limpos = [str(m) for m in matches_bloco if m]
                    termos_da_linha.extend(list(set(termos_limpos)))

            # 3. Cálculo da nota de corte
            porcentagem_sucesso = (blocos_com_match / total_blocos) * 100
            #print(f"Possui:", blocos_com_match)
            #print(f"Porcentagem de sucesso:", porcentagem_sucesso)
            if porcentagem_sucesso >= corte_porcentagem:
                df.at[idx, "temas_encontrados"].append(f"{tema_codigo} ({porcentagem_sucesso:.1f}%)")
                termos_unicos = list(set([t for t in termos_da_linha if t.strip()]))
                df.at[idx, "termos_capturados"].append(f"{tema_codigo}: {termos_unicos}")
                if df.at[idx,"TEMA_CODIGO"] == str(tema_codigo):
                    df.at[idx, "acertou_tema"] = True


    return df

In [19]:
pd.set_option('display.max_colwidth', 512)  # Shows full cell contents without cutting off text


In [20]:
df_lematizado = pd.read_parquet(r"C:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification-1\data\dataset_enriquecido_10062026_lematizado.parquet")
df_temas = pd.read_csv(r"C:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification-1\data\Temas_STF_com_Regex.csv")

In [21]:
palavras_1 = pd.read_csv(r"C:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification-1\data\Palavras chaves temas do STF - Leonardo.csv")
palavras_2 = pd.read_csv(r"C:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification-1\data\Palavras chaves temas do STJ - Patrícia.csv")
palavras = pd.concat([palavras_1,palavras_2])

In [22]:
palavras['PALAVRAS_LEMATIZADAS'] = palavras['PALAVRAS-CHAVES'].fillna("").apply(lambda doc: " ".join([token.lemma_ for token in nlp(doc)])) 

In [23]:
palavras['PALAVRAS_LEMATIZADAS_REGEX'] = palavras['PALAVRAS_LEMATIZADAS'].apply(lambda texto: converter_expressao_para_regex(texto))

In [24]:
df_lematizado_sample = df_lematizado.sample(frac=0.01, random_state=42)
df_sample = df_lematizado.iloc[[0]]

In [25]:
palavras.info()

<class 'pandas.DataFrame'>
Index: 160 entries, 0 to 93
Data columns (total 6 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   TEMA
ORIGEM                 160 non-null    str  
 1   TEMA CÓDIGO                 160 non-null    int64
 2   PALAVRAS-CHAVES             100 non-null    str  
 3   SITUAÇÃO/OBSERVAÇÕES        101 non-null    str  
 4   PALAVRAS_LEMATIZADAS        160 non-null    str  
 5   PALAVRAS_LEMATIZADAS_REGEX  160 non-null    str  
dtypes: int64(1), str(5)
memory usage: 673.5 KB


In [ ]:
palavras.fillna({"PALAVRAS_LEMATIZADAS_REGEX": "NAN"}, inplace=True)
df_lematizado_sample_with_regex = aplicar_regex_flexivel_no_dataframe(df_lematizado, palavras)

In [ ]:
df_lematizado_sample_with_regex.to_parquet("df_lematizado_sample_with_regex.parquet")

In [ ]:
import multiprocessing_regex as mr

In [ ]:
df_multiprocessado = mr.aplicar_regex_flexivel_no_dataframe_multiprocessing(df_lematizado, palavras)

In [ ]:
df_multiprocessado.to_parquet("df_multiprocessado.parquet")